In [2]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer # Text -> Numeric
from sklearn.cluster import KMeans
from umap import UMAP # Dimensionality Reduction for Visualization
import plotly.express as px  # Interactive Visualization

c:\Users\THADLOW\AppData\Local\anaconda3\envs\is455_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# 1. Gather review/text data
kol_posts = pd.read_csv("../data/kol/KOL_Posts.csv")
rest_reviews = pd.read_parquet("../_3_marketing/restaurant_reviews.parquet")
rest_best_reviews = pd.read_parquet("../_3_marketing/restaurant_reviews_all_best.parquet")
rest = pd.read_parquet("../data/mv_dataset_parquet/restaurants.parquet")
placesapinew = pd.read_csv("../_1_eda/places_api_new_results.csv")

#rest[rest['name'] == 'Melody Bangkok']
#kol_posts[kol_posts['Restaurant Name'] == 'Audrey Cafe Thonglor Soi 11']
rest

,restaurant_id,name,days_in_advance
0,33,Audrey Cafe Thonglor Soi 11,90
1,168,Attico Cucina Italiana Radisson Blu Plaza Bangkok,60
2,220,The Living Room at Sheraton Grande Sukhumvit A...,30
3,222,Rain Tree Cafe at The Athenee Hotel,90
4,273,Skyline at Avani+ Riverside Bangkok Hotel,90
...,...,...,...
2469,6937,Vista Restaurant at Avista Hideaway Phuket Pat...,90
2470,6938,Lamai Himnam (Chiang Mai),90
2471,6940,Sun's Cafe at Hotel Grand Pacific Singapore,90
2472,6941,Food Lala Thai Cuisine,90


In [4]:
# Aggregate KOL Posts per restaurant
kol_aggregated = (
    kol_posts.groupby('Restaurant Name')['Content']
    .apply(lambda x: ' '.join(x.dropna().astype(str)))
    .reset_index()
)
kol_aggregated.columns = ['Restaurant Name', 'kol_text']

#print(kol_aggregated)

placesapinew['google_text'] = ( # Combine relevant fields from placesapinew into one text string for each restaurant
        placesapinew['input_string'].fillna('') + ' ' +
        placesapinew['raw_types'].fillna('').str.replace(',', ' ') + ' ' +
        placesapinew['Cuisine'].fillna('')
)

# Merge KOL aggregated text with placesapinew to get a combined dataset for clustering
kol_restaurant_mapping = kol_posts[['Restaurant Name', 'Restaurant Code']].drop_duplicates()
places_merged = placesapinew.merge(
    kol_restaurant_mapping,
    left_on='input_string', # Assuming 'input_string' in placesapinew corresponds to 'Restaurant Name' in kol_posts
    right_on='Restaurant Name', # Merge on restaurant name
    how='left'
)

# Merge dataset with both the google text and the KOL text for each restaurant
places_aggregated = places_merged[['Restaurant Name', 'google_text']].rename(
    columns={'Restaurant Name': 'restaurant name'}
)

rest_reviews['rest_reviews'] = ( # Combine restaurant name and review text into one string for clustering
    rest_reviews['input_restaurant_name'].fillna('') + ' ' +
    rest_reviews['review_text'].fillna('').str.replace(',', ' ') 
)

rest_places_merged = rest_reviews.merge( # Merge restaurant reviews with the aggregated places data to get a combined dataset for clustering
    places_aggregated,
    left_on='input_restaurant_name', # Assuming 'input_string' in rest_reviews corresponds to 'restaurant name' in places_aggregated
    right_on='restaurant name', # Merge on restaurant name
    how='left'
)

places_aggregated.head()
rest_places_merged.head()

,input_restaurant_name,pulled_at_utc,source,place_id,matched_name,review_id,author_name,author_url,profile_photo_url,rating,language,review_text,relative_time_description,review_time_unix,review_time_utc,rest_reviews,restaurant name,google_text
0,Melody Bangkok,2026-02-16T15:06:12.727968+00:00,google_places,ChIJ-6r24ICf4jARC6jtWZYj7S8,Melody Bangkok,ChIJ-6r24ICf4jARC6jtWZYj7S8_1770211654_NATTAWA...,NATTAWAT RATTANAPONGPRAKIT,https://www.google.com/maps/contrib/1024293807...,https://lh3.googleusercontent.com/a-/ALV-UjVOX...,5,en-US,We ate it all before I could take a picture! T...,a week ago,1770211654,2026-02-04T13:27:34+00:00,Melody Bangkok We ate it all before I could ta...,NaN,NaN
1,Melody Bangkok,2026-02-16T15:06:12.727968+00:00,google_places,ChIJ-6r24ICf4jARC6jtWZYj7S8,Melody Bangkok,ChIJ-6r24ICf4jARC6jtWZYj7S8_1770021873_Mizutan...,Mizutani Koichi,https://www.google.com/maps/contrib/1102947333...,https://lh3.googleusercontent.com/a-/ALV-UjU8O...,1,en-US,The Happy Hour menu does not include draft bee...,2 weeks ago,1770021873,2026-02-02T08:44:33+00:00,Melody Bangkok The Happy Hour menu does not in...,NaN,NaN
2,Melody Bangkok,2026-02-16T15:06:12.727968+00:00,google_places,ChIJ-6r24ICf4jARC6jtWZYj7S8,Melody Bangkok,ChIJ-6r24ICf4jARC6jtWZYj7S8_1769797681_Patpoom...,Patpoom MSoulScent,https://www.google.com/maps/contrib/1168736595...,https://lh3.googleusercontent.com/a-/ALV-UjVEH...,5,en-US,"The staff are nice, friendly, and polite.\nThe...",2 weeks ago,1769797681,2026-01-30T18:28:01+00:00,Melody Bangkok The staff are nice friendly a...,NaN,NaN
3,Melody Bangkok,2026-02-16T15:06:12.727968+00:00,google_places,ChIJ-6r24ICf4jARC6jtWZYj7S8,Melody Bangkok,ChIJ-6r24ICf4jARC6jtWZYj7S8_1769692334_Jesada ...,Jesada Bobpahow,https://www.google.com/maps/contrib/1041664502...,https://lh3.googleusercontent.com/a-/ALV-UjVw9...,5,en-US,"Delicious food, great atmosphere.",2 weeks ago,1769692334,2026-01-29T13:12:14+00:00,Melody Bangkok Delicious food great atmosphere.,NaN,NaN
4,Melody Bangkok,2026-02-16T15:06:12.727968+00:00,google_places,ChIJ-6r24ICf4jARC6jtWZYj7S8,Melody Bangkok,ChIJ-6r24ICf4jARC6jtWZYj7S8_1769692169_อรทัย ห...,อรทัย หลงทัพไทย,https://www.google.com/maps/contrib/1091362972...,https://lh3.googleusercontent.com/a/ACg8ocLqF_...,5,en-US,"The taste is good, the food is delicious.",2 weeks ago,1769692169,2026-01-29T13:09:29+00:00,Melody Bangkok The taste is good the food is ...,NaN,NaN


In [5]:
# 2. Prepare text data

rest_places_aggregated = ( # Aggregate reviews and google text by restaurant name to get one combined text string per restaurant for clustering
    rest_places_merged
    .groupby('input_restaurant_name')[['rest_reviews', 'google_text']]  # use double brackets
    .agg({
        'rest_reviews': lambda x: ' '.join(x.dropna().astype(str)),
        'google_text': lambda x: ' '.join(x.dropna().astype(str))
    })
    .reset_index()
    .rename(columns={'input_restaurant_name': 'restaurant name'})
)

# Combine both rest reviews and google text columns into one
rest_places_aggregated['text'] = (
    rest_places_aggregated['rest_reviews'].fillna('') + ' ' +
    rest_places_aggregated['google_text'].fillna('')
)

reviews = rest_places_aggregated[['restaurant name', 'text']].copy() # Final dataset for clustering with restaurant name and combined text

reviews

,restaurant name,text
0,&T modern Thai grill,&T modern Thai grill -The restaurant is beauti...
1,'@RICE Restaurant by At Rice Resort (Nakhon Na...,'@RICE Restaurant by At Rice Resort (Nakhon Na...
2,100 Degrees Hotpot Central Chaengwattana,100 Degrees Hotpot Central Chaengwattana Delic...
3,100 Degrees Hotpot Cosmo Bazaar,100 Degrees Hotpot Cosmo Bazaar Kimchi soup is...
4,123 ICHI NI SAN Sathorn Soi 1,123 ICHI NI SAN Sathorn Soi 1 123 ICHI NI SAN...
...,...,...
1942,sala ayutthaya staycation (Ayutthaya),sala ayutthaya staycation (Ayutthaya) sala ay...
1943,sala bang pa-in Staycation (Ayutthaya),sala bang pa-in Staycation (Ayutthaya) This st...
1944,อร่อย Together,อร่อย Together Went to sing karaoke and was pl...
1945,中国好味道 China Taste,中国好味道 China Taste Food is relatively less salt...


In [6]:
print("Unique KOL restaurants:", kol_posts['Restaurant Name'].nunique())
print("After merge:", places_merged['Restaurant Name'].nunique())

print("Unique review restaurants:", rest_reviews['input_restaurant_name'].nunique())
print("Final merged:", rest_places_merged['input_restaurant_name'].nunique())

Unique KOL restaurants: 301
After merge: 197
Unique review restaurants: 1947
Final merged: 1947


In [7]:
# 3. Text vectorization
vectorizer = TfidfVectorizer( # Measures importance of words in each document relative to the corpus
    max_features=200, # Limit to top 200 words to reduce noise and dimensionality
    ngram_range=(1, 2), # Capture both single words and common two-word phrases (bigrams)
    stop_words='english', # Remove common English words that don't add much meaning
    min_df=2 # Only keep words that appear in at least 2 documents to filter out very rare terms
)
text_vectors = vectorizer.fit_transform(reviews['text']) # Convert the text data into a numeric matrix where rows are documents and columns are the importance of each word/phrase

In [8]:
# Cluster Posts (Initial Clustering at Post Level)
print("\n" + "="*70)
print("STEP 3: Clustering Posts (K-Means with 5 Clusters)")
print("="*70)

kmeans = KMeans(n_clusters=7, random_state=42, n_init=10)
clusters = kmeans.fit_predict(text_vectors)
reviews['cluster'] = clusters

print(f"Assigned {len(reviews)} posts to 7 clusters")
print("\nPost distribution across clusters:")
print(reviews['cluster'].value_counts().sort_index())


STEP 3: Clustering Posts (K-Means with 5 Clusters)
Assigned 1947 posts to 7 clusters

Post distribution across clusters:
cluster
0    954
1    177
2    180
3    151
4    101
5    150
6    234
Name: count, dtype: int64


In [22]:
reviews.to_csv("reviews.csv")

In [9]:
# Define Predefined Themes
cluster_themes = {
    0: "Various Menus & Quality Food",
    1: "Restaurant with Good View",
    2: "Attentive Service but Pricey",
    3: "Good Location",
    4: "Extensive Alcohol Beverage Options"
}

reviews['theme'] = reviews['cluster'].map(cluster_themes)

In [10]:
# Validate Cluster Themes (Check Keywords)
print("\n" + "="*70)
print("STEP 4: Validating Cluster Themes (Auto vs Manual)")
print("="*70)

for cluster_id in range(5):
    cluster_texts = reviews[reviews['cluster'] == cluster_id]['text']
    
    if len(cluster_texts) > 0:
        cluster_vectorizer = TfidfVectorizer(max_features=10, stop_words='english')
        cluster_vectorizer.fit(cluster_texts)
        top_terms = cluster_vectorizer.get_feature_names_out()
        
        print(f"\n Cluster {cluster_id} → {cluster_themes[cluster_id]}")
        print(f"    Posts: {len(cluster_texts)} ({len(cluster_texts)/len(reviews)*100:.1f}%)")
        print(f"    Auto-detected keywords: {', '.join(top_terms)}")
        print(f"    Check if keywords match your theme name!")

# MULTI-THEME RESTAURANT ASSIGNMENT
print("\n" + "="*70)
print("STEP 5: Aggregating to Restaurant Level (Multi-Theme Method)")
print("="*70)

# Count posts per restaurant per cluster
restaurant_theme_counts = (
    reviews.groupby(['restaurant name', 'cluster', 'theme'])
    .size()
    .reset_index(name='post_count')
)

# Get total posts per restaurant
restaurant_post_totals = reviews.groupby('restaurant name').size().reset_index(name='total_posts')

# Merge to calculate percentages
restaurant_theme_counts = restaurant_theme_counts.merge(
    restaurant_post_totals, 
    on='restaurant name'
)

restaurant_theme_counts['percentage'] = ( # Calculate percentage of posts for each theme per restaurant
    restaurant_theme_counts['post_count'] / restaurant_theme_counts['total_posts'] * 100
)

print(f"\n Analyzing {len(restaurant_post_totals)} unique restaurants...")


STEP 4: Validating Cluster Themes (Auto vs Manual)

 Cluster 0 → Various Menus & Quality Food
    Posts: 954 (49.0%)
    Auto-detected keywords: atmosphere, delicious, excellent, food, good, great, like, restaurant, service, staff
    Check if keywords match your theme name!

 Cluster 1 → Restaurant with Good View
    Posts: 177 (9.1%)
    Auto-detected keywords: delicious, food, good, hin, hua, pattaya, phuket, resort, restaurant, service
    Check if keywords match your theme name!

 Cluster 2 → Attentive Service but Pricey
    Posts: 180 (9.2%)
    Auto-detected keywords: buffet, central, delicious, food, good, rama, restaurant, service, shabu, staff
    Check if keywords match your theme name!

 Cluster 3 → Good Location
    Posts: 151 (7.8%)
    Auto-detected keywords: atmosphere, bangkok, bar, food, good, great, restaurant, rooftop, service, staff
    Check if keywords match your theme name!

 Cluster 4 → Extensive Alcohol Beverage Options
    Posts: 101 (5.2%)
    Auto-detected

In [11]:
# Keep themes with at least 20% of restaurant's posts
THRESHOLD = 20  # Adjust this threshold (e.g., 15, 20, 25, 30)

significant_themes = restaurant_theme_counts[
    restaurant_theme_counts['percentage'] >= THRESHOLD
].copy()

print(f"\n Keeping themes that represent ≥{THRESHOLD}% of a restaurant's posts")
print(f"   Found {len(significant_themes)} restaurant-theme combinations")

# Create multi-theme labels
restaurant_multi_themes = (
    significant_themes.groupby('restaurant name')
    .agg({
        'theme': lambda x: ' + '.join(sorted(x)),  # Combine themes
        'cluster': lambda x: list(x),  # Keep cluster IDs
        'percentage': lambda x: list(x)  # Keep percentages
    })
    .reset_index()
)

restaurant_multi_themes.columns = ['restaurant name', 'combined_themes', 'clusters', 'percentages']

print("\n" + "="*70)
print("RESTAURANT THEME DISTRIBUTION (Multi-Theme Assignment)")
print("="*70)
print(restaurant_multi_themes['combined_themes'].value_counts().head(15))

# Also create primary theme (for visualization)
# Assign each restaurant to its DOMINANT theme (highest percentage)
restaurant_primary_theme = (
    restaurant_theme_counts
    .sort_values('percentage', ascending=False)
    .groupby('restaurant name')
    .first()
    .reset_index()
)

restaurant_primary_theme = restaurant_primary_theme[[
    'restaurant name', 'cluster', 'theme', 'percentage', 'post_count', 'total_posts'
]]

print("\n" + "="*70)
print("PRIMARY THEME DISTRIBUTION (Dominant Theme per Restaurant)")
print("="*70)
print(restaurant_primary_theme['theme'].value_counts())


 Keeping themes that represent ≥20% of a restaurant's posts
   Found 1563 restaurant-theme combinations

RESTAURANT THEME DISTRIBUTION (Multi-Theme Assignment)
combined_themes
Various Menus & Quality Food          954
Attentive Service but Pricey          180
Restaurant with Good View             177
Good Location                         151
Extensive Alcohol Beverage Options    101
Name: count, dtype: int64

PRIMARY THEME DISTRIBUTION (Dominant Theme per Restaurant)
theme
Various Menus & Quality Food          954
Attentive Service but Pricey          180
Restaurant with Good View             177
Good Location                         151
Extensive Alcohol Beverage Options    101
Name: count, dtype: int64


In [12]:
# STEP 7: Create Restaurant-Level Embeddings for Visualization
print("\n" + "="*70)
print("STEP 6: Creating Restaurant-Level Embeddings")
print("="*70)

restaurant_embeddings = []
restaurant_names_list = []

# Reset index to ensure alignment with text_vectors
reviews_reset = reviews.reset_index(drop=True)

for rest_id in restaurant_primary_theme['restaurant name']:
    # Get positions (not indices) of posts for this restaurant
    post_positions = reviews_reset[reviews_reset['restaurant name'] == rest_id].index.tolist()
    
    if len(post_positions) == 0:
        continue
    
    # Average their TF-IDF vectors using positions
    rest_vector = text_vectors[post_positions].mean(axis=0)
    restaurant_embeddings.append(np.array(rest_vector).flatten())
    restaurant_names_list.append(rest_id)

restaurant_embeddings = np.array(restaurant_embeddings)
print(f" Created embeddings for {len(restaurant_embeddings)} restaurants")


STEP 6: Creating Restaurant-Level Embeddings
 Created embeddings for 1563 restaurants


In [13]:
# STEP 8: Dimensionality Reduction (200D → 2D using UMAP)
print("\n" + "="*70)
print("STEP 7: Reducing Dimensions (200D → 2D for Visualization)")
print("="*70)

# Can experiment with UMAP parameters like n_neighbors and min_dist to see how it affects the clustering in 2D space
reducer = UMAP(
    n_components=2,
    n_neighbors=30,    # ← Increase from 15 (preserves more global structure)
    min_dist=1,      # ← Increase from 0.1 (allows more spread)
    metric='cosine',   # ← Use cosine similarity (better for text)
    random_state=42
)
embeddings_2d = reducer.fit_transform(restaurant_embeddings)
print(f" Reduced to 2D coordinates")


STEP 7: Reducing Dimensions (200D → 2D for Visualization)


c:\Users\THADLOW\AppData\Local\anaconda3\envs\is455_env\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


 Reduced to 2D coordinates


In [21]:
viz_df.to_csv("viz_df.csv")

In [14]:
# Create Visualization DataFrame
viz_df = pd.DataFrame({
    'restaurant name': restaurant_names_list, # Align restaurant names with the order of embeddings
    'UMAP Component 1 (Semantic Similarity Axis)': embeddings_2d[:, 0], # UMAP component 1 captures the main semantic differences between restaurants based on their reviews and google text
    'UMAP Component 2 (Theme Variation Axis)': embeddings_2d[:, 1], # UMAP component 2 captures the variation in themes among restaurants
})

# Merge with primary theme (for color)
viz_df = viz_df.merge( # Merge primary theme info for coloring the points in the visualization
    restaurant_primary_theme[['restaurant name', 'cluster', 'theme', 'total_posts', 'percentage']], 
    on='restaurant name'
)

# Merge with multi-themes (for hover info)
viz_df = viz_df.merge(
    restaurant_multi_themes[['restaurant name', 'combined_themes']], 
    on='restaurant name',
    how='left'
)

# Fill single-theme restaurants
viz_df['combined_themes'] = viz_df['combined_themes'].fillna(viz_df['theme'])

# Rename for clarity
viz_df = viz_df.rename(columns={
    'theme': 'Primary Theme',
    'total_posts': 'Number of Posts',
    'percentage': 'Primary Theme %',
    'combined_themes': 'All Themes'
})

print(f"\n Prepared {len(viz_df)} restaurants for visualization")


 Prepared 1563 restaurants for visualization


In [15]:
# Create Interactive Plotly Visualization
print("\n" + "="*70)
print("STEP 8: Creating Interactive Visualization")
print("="*70)

fig = px.scatter( # Create an interactive scatter plot with Plotly Express
    viz_df,
    x='UMAP Component 1 (Semantic Similarity Axis)',
    y='UMAP Component 2 (Theme Variation Axis)',
    color='Primary Theme',
    size='Number of Posts',  # Bigger dots = more posts about this restaurant
    hover_data={ # Show these details when hovering over a restaurant
        'restaurant name': True,
        'Primary Theme': True,
        'All Themes': True,  # Shows multi-theme assignments
        'Number of Posts': True,
        'Primary Theme %': ':.1f',
        'UMAP Component 1 (Semantic Similarity Axis)': ':.2f',
        'UMAP Component 2 (Theme Variation Axis)': ':.2f'
    },
    title='<b>Restaurant Segmentation by Customer Perception Themes</b><br>' + 
          '<sub>(1 Dot = 1 Restaurant | Size = Number of Posts | Color = Primary Theme)</sub>',
    color_discrete_sequence=px.colors.qualitative.Set3,
)

fig.update_layout( # Customize layout for better readability and aesthetics
    font=dict(size=12),
    title_font=dict(size=16),
    width=1400,
    height=800,
    legend=dict(
        title="Primary Customer Perception Theme",
        orientation="v",
        yanchor="top",
        y=1,
        xanchor="left",
        x=1.02
    ),
    hovermode='closest'
)

# Update marker size range for better visibility
fig.update_traces(
    marker=dict(
        sizemode='diameter',
        sizemin=5,
        sizeref=2.*max(viz_df['Number of Posts'])/(40.**2),
        line=dict(width=1, color='DarkSlateGrey')
    )
)

# Add cluster centroids with labels
centroids = viz_df.groupby("Primary Theme")[
    ['UMAP Component 1 (Semantic Similarity Axis)', 
     'UMAP Component 2 (Theme Variation Axis)']
].mean().reset_index()

for _, row in centroids.iterrows():
    fig.add_annotation(
        x=row['UMAP Component 1 (Semantic Similarity Axis)'],
        y=row['UMAP Component 2 (Theme Variation Axis)'],
        text=f"<b>{row['Primary Theme']}</b>",
        showarrow=False,
        font=dict(size=10, color="black"),
        bgcolor="rgba(255, 255, 255, 0.85)",
        bordercolor="black",
        borderwidth=1.5,
        borderpad=5
    )

fig.write_html("restaurant_clusters_multi_theme.html")
print("\n Visualization saved to 'restaurant_clusters_multi_theme.html'")


STEP 8: Creating Interactive Visualization

 Visualization saved to 'restaurant_clusters_multi_theme.html'


In [16]:
# Save Results to CSV
print("\n" + "="*70)
print("STEP 9: Saving Results")
print("="*70)

# Save restaurant-level data with all themes
output_df = viz_df[[
    'restaurant name', 
    'Primary Theme', 
    'All Themes', 
    'Number of Posts',
    'Primary Theme %'
]].copy()

output_df.to_csv('restaurants_with_multi_themes.csv', index=False)
print(" Saved 'restaurants_with_multi_themes.csv'")

# Also save detailed theme breakdown
restaurant_theme_counts.to_csv('restaurant_theme_details.csv', index=False)
print(" Saved 'restaurant_theme_details.csv' (shows % for each theme)")


STEP 9: Saving Results
 Saved 'restaurants_with_multi_themes.csv'
 Saved 'restaurant_theme_details.csv' (shows % for each theme)


In [17]:
# Summary Statistics
print("\n" + "="*70)
print(" SUMMARY STATISTICS")
print("="*70)

print(f"\n Total Restaurants: {len(viz_df)}")
print(f" Total Posts: {viz_df['Number of Posts'].sum()}")
print(f" Avg Posts per Restaurant: {viz_df['Number of Posts'].mean():.1f}")

print("\n Single-Theme vs Multi-Theme Restaurants:")
single_theme = viz_df[viz_df['All Themes'] == viz_df['Primary Theme']]
multi_theme = viz_df[viz_df['All Themes'] != viz_df['Primary Theme']]

print(f"   Single-theme restaurants: {len(single_theme)} ({len(single_theme)/len(viz_df)*100:.1f}%)")
print(f"   Multi-theme restaurants: {len(multi_theme)} ({len(multi_theme)/len(viz_df)*100:.1f}%)")

print("\n Most Common Multi-Theme Combinations:")
if len(multi_theme) > 0:
    print(multi_theme['All Themes'].value_counts().head(10))

print("\n CLUSTERING COMPLETE with new data!")
print("="*70)


 SUMMARY STATISTICS

 Total Restaurants: 1563
 Total Posts: 1563
 Avg Posts per Restaurant: 1.0

 Single-Theme vs Multi-Theme Restaurants:
   Single-theme restaurants: 1563 (100.0%)
   Multi-theme restaurants: 0 (0.0%)

 Most Common Multi-Theme Combinations:

 CLUSTERING COMPLETE with new data!


In [18]:
# Merge with restaurant to get restaurant id
clustering_results = pd.merge(
    single_theme,
    rest,
    left_on="restaurant name",
    right_on="name",
    how="inner"
)

clustering_results.to_csv('clustering_results.csv', index=False)

In [19]:
clustering_results.shape

(1373, 11)

In [20]:
clustering_results['restaurant_id'].nunique()

1373

In [ ]:
# STEP 10: Export dashboard artifacts for Streamlit
from pathlib import Path
import numpy as np
import pandas as pd

print("\n" + "="*70)
print("STEP 10: Export dashboard artifacts for Streamlit")
print("="*70)

def _find_project_root() -> Path:
    for path in [Path.cwd(), *Path.cwd().parents]:
        if (path / "frontend_dashboard").exists():
            return path
    return Path.cwd()

project_root = _find_project_root()
export_dir = project_root / "frontend_dashboard" / "data" / "clustering"
export_dir.mkdir(parents=True, exist_ok=True)

# 1) restaurant_cluster_assignments
if "clustering_results" in globals() and isinstance(clustering_results, pd.DataFrame) and len(clustering_results):
    assignments = clustering_results.copy()
elif "viz_df" in globals() and isinstance(viz_df, pd.DataFrame) and len(viz_df):
    assignments = viz_df.copy()
else:
    assignments = pd.read_csv("clustering_results.csv") if Path("clustering_results.csv").exists() else pd.DataFrame()

if not assignments.empty:
    if "restaurant name" in assignments.columns and "name" in assignments.columns:
        assignments["name"] = assignments["name"].fillna(assignments["restaurant name"])
        assignments = assignments.drop(columns=["restaurant name"])

    rename_map = {
        "restaurant name": "name",
        "cluster": "cluster_id",
        "hybrid_cluster": "cluster_id",
        "Primary Theme": "cluster_label",
        "theme": "cluster_label",
        "UMAP Component 1 (Semantic Similarity Axis)": "x",
        "UMAP Component 2 (Theme Variation Axis)": "y",
    }
    assignments = assignments.rename(columns=rename_map)

    if "cluster_id" not in assignments.columns:
        assignments["cluster_id"] = 0
    if "cluster_label" not in assignments.columns:
        assignments["cluster_label"] = assignments["cluster_id"].apply(lambda v: f"Cluster {v}")
    if "cluster_confidence" not in assignments.columns:
        assignments["cluster_confidence"] = np.nan
    if "year_month" not in assignments.columns:
        assignments["year_month"] = pd.NaT

    keep_cols = [
        c for c in [
            "restaurant_id", "name", "cluster_id", "cluster_label",
            "x", "y", "cluster_confidence", "year_month"
        ] if c in assignments.columns
    ]
    assignments = assignments[keep_cols].copy()

# Ensure full coverage: every momentum restaurant has a clustering category
momentum_path = project_root / "_2_feature_engineering+momentum" / "start" / "restaurants_agg_performance.parquet"
if momentum_path.exists() and not assignments.empty:
    momentum_latest = (
        pd.read_parquet(momentum_path)
        .sort_values("year_month")
        .groupby("name", as_index=False)
        .last()
    )

    assignments["name"] = assignments["name"].astype(str).str.strip()
    assignments["name_norm"] = assignments["name"].str.lower().str.replace(r"\s+", " ", regex=True).str.strip()

    momentum_latest["name"] = momentum_latest["name"].astype(str).str.strip()
    momentum_latest["name_norm"] = momentum_latest["name"].str.lower().str.replace(r"\s+", " ", regex=True).str.strip()

    if "restaurant_id" in assignments.columns:
        assignments["restaurant_id"] = pd.to_numeric(assignments["restaurant_id"], errors="coerce")
    if "restaurant_id" in momentum_latest.columns:
        momentum_latest["restaurant_id"] = pd.to_numeric(momentum_latest["restaurant_id"], errors="coerce")

    if "restaurant_id" in assignments.columns and "restaurant_id" in momentum_latest.columns and assignments["restaurant_id"].notna().any():
        existing_ids = set(assignments["restaurant_id"].dropna().astype(int))
        missing = momentum_latest[~momentum_latest["restaurant_id"].fillna(-1).astype(int).isin(existing_ids)].copy()
    else:
        missing = momentum_latest[~momentum_latest["name_norm"].isin(set(assignments["name_norm"]))].copy()

    if len(missing):
        x_anchor = float(pd.to_numeric(assignments["x"], errors="coerce").min()) - 1.6 if "x" in assignments.columns and pd.to_numeric(assignments["x"], errors="coerce").notna().any() else -1.6
        y_anchor = float(pd.to_numeric(assignments["y"], errors="coerce").min()) - 1.6 if "y" in assignments.columns and pd.to_numeric(assignments["y"], errors="coerce").notna().any() else -1.6

        n_missing = len(missing)
        angles = np.linspace(0, 2 * np.pi, n_missing, endpoint=False)
        radius = np.linspace(0.08, 0.28, n_missing)

        missing = missing.rename(columns={"name": "name"}).copy()
        missing["cluster_id"] = -1
        missing["cluster_label"] = "Unclustered - no clustering text"
        missing["cluster_confidence"] = 0.0
        missing["x"] = x_anchor + radius * np.cos(angles)
        missing["y"] = y_anchor + radius * np.sin(angles)
        missing["year_month"] = pd.NaT

        keep_all = [
            c for c in [
                "restaurant_id", "name", "cluster_id", "cluster_label",
                "x", "y", "cluster_confidence", "year_month", "name_norm"
            ] if c in assignments.columns or c in missing.columns
        ]
        assignments = assignments.reindex(columns=keep_all)
        missing = missing.reindex(columns=keep_all)
        assignments = pd.concat([assignments, missing], ignore_index=True)

    assignments["cluster_id"] = pd.to_numeric(assignments["cluster_id"], errors="coerce").fillna(-1).astype(int)
    assignments.loc[assignments["cluster_id"] == -1, "cluster_label"] = "Unclustered - no clustering text"
    assignments = assignments.drop(columns=["name_norm"], errors="ignore")

assignments_path = export_dir / "restaurant_cluster_assignments.parquet"
assignments.to_parquet(assignments_path, index=False)
print(f"Saved: {assignments_path} ({len(assignments):,} rows)")

# 2) cluster_keywords
if "restaurant_theme_counts" in globals() and isinstance(restaurant_theme_counts, pd.DataFrame) and len(restaurant_theme_counts):
    keywords = (
        restaurant_theme_counts
        .rename(columns={"cluster": "cluster_id", "theme": "keyword", "post_count": "weight"})
        [["cluster_id", "keyword", "weight"]]
        .copy()
    )
    keywords["weight"] = pd.to_numeric(keywords["weight"], errors="coerce").fillna(0)
    keywords = (
        keywords.sort_values(["cluster_id", "weight"], ascending=[True, False])
        .groupby("cluster_id")
        .head(50)
        .copy()
    )
    keywords["rank"] = keywords.groupby("cluster_id").cumcount() + 1
else:
    tmp_reviews = reviews.copy() if "reviews" in globals() else (
        pd.read_csv("reviews.csv") if Path("reviews.csv").exists() else pd.DataFrame()
    )
    if not tmp_reviews.empty:
        tmp_reviews = tmp_reviews.rename(columns={"cluster": "cluster_id", "text": "raw_text"})
        rows = []
        for cluster_id, cdf in tmp_reviews.groupby("cluster_id"):
            tokens = (
                cdf["raw_text"].fillna("").astype(str)
                .str.lower()
                .str.findall(r"[a-zA-Z]{3,}")
                .explode()
                .dropna()
            )
            top_terms = tokens.value_counts().head(50)
            for rank, (word, weight) in enumerate(top_terms.items(), start=1):
                rows.append({"cluster_id": cluster_id, "keyword": word, "weight": float(weight), "rank": rank})
        keywords = pd.DataFrame(rows)
    else:
        keywords = pd.DataFrame(columns=["cluster_id", "keyword", "weight", "rank"])

keywords_path = export_dir / "cluster_keywords.parquet"
keywords.to_parquet(keywords_path, index=False)
print(f"Saved: {keywords_path} ({len(keywords):,} rows)")

# 3) restaurant_text_corpus
if "reviews" in globals() and isinstance(reviews, pd.DataFrame) and len(reviews):
    text_corpus = reviews.copy()
elif Path("reviews.csv").exists():
    text_corpus = pd.read_csv("reviews.csv")
else:
    text_corpus = pd.DataFrame(columns=["restaurant name", "text", "cluster"])

text_corpus = text_corpus.rename(columns={"restaurant name": "name", "text": "raw_text", "cluster": "cluster_id"})
if "text_id" not in text_corpus.columns:
    text_corpus["text_id"] = np.arange(1, len(text_corpus) + 1)
if "clean_text" not in text_corpus.columns:
    text_corpus["clean_text"] = (
        text_corpus["raw_text"].fillna("").astype(str)
        .str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
    )
if "year_month" not in text_corpus.columns:
    text_corpus["year_month"] = pd.NaT

text_cols = [c for c in ["name", "text_id", "raw_text", "clean_text", "cluster_id", "year_month"] if c in text_corpus.columns]
text_corpus = text_corpus[text_cols].copy()

text_path = export_dir / "restaurant_text_corpus.parquet"
text_corpus.to_parquet(text_path, index=False)
print(f"Saved: {text_path} ({len(text_corpus):,} rows)")

# 4) cluster_strategy_outcomes
marketing_path = project_root / "_3_marketing" / "activity_performance_with_roi.csv"
if marketing_path.exists() and not assignments.empty and "restaurant_id" in assignments.columns:
    marketing = pd.read_csv(marketing_path)
    marketing["restaurant_id"] = pd.to_numeric(marketing.get("restaurant_id"), errors="coerce")
    assign_join = assignments[[c for c in ["restaurant_id", "name", "cluster_id", "cluster_label"] if c in assignments.columns]].copy()
    assign_join["restaurant_id"] = pd.to_numeric(assign_join.get("restaurant_id"), errors="coerce")
    assign_join = assign_join.dropna(subset=["restaurant_id"]).drop_duplicates("restaurant_id")

    strategy = marketing.merge(assign_join, on="restaurant_id", how="inner", suffixes=("", "_cluster"))

    def _strategy_name(row: pd.Series) -> str:
        channel = str(row.get("channel", "Unknown")).upper()
        if channel == "CRM":
            topic = str(row.get("crm_topic", "")).strip()
            campaign = str(row.get("crm_campaign_name", "")).strip()
            return f"CRM | {topic or campaign or 'campaign'}"
        if channel == "FB":
            campaign = str(row.get("fb_campaign", "")).strip()
            return f"FB | {campaign or 'campaign'}"
        if channel == "KOL":
            user = str(row.get("kol_username", "")).strip()
            return f"KOL | @{user}" if user else "KOL | creator"
        return f"{channel} | campaign"

    strategy["strategy_name"] = strategy.apply(_strategy_name, axis=1)
    strategy["bookings_before"] = pd.to_numeric(strategy.get("bookings_baseline"), errors="coerce")
    strategy["bookings_after"] = pd.to_numeric(strategy.get("bookings_during"), errors="coerce")
    strategy["incremental_revenue_thb"] = pd.to_numeric(strategy.get("incremental_revenue_thb"), errors="coerce")
    strategy["revenue_after"] = pd.to_numeric(strategy.get("total_campaign_revenue"), errors="coerce")
    strategy["revenue_before"] = strategy["revenue_after"] - strategy["incremental_revenue_thb"]
    strategy["bookings_uplift_pct"] = np.where(
        strategy["bookings_before"] > 0,
        (strategy["bookings_after"] - strategy["bookings_before"]) / strategy["bookings_before"],
        np.nan,
    )
    strategy["revenue_uplift_pct"] = np.where(
        strategy["revenue_before"] > 0,
        strategy["incremental_revenue_thb"] / strategy["revenue_before"],
        np.nan,
    )
    strategy["roi"] = pd.to_numeric(strategy.get("roi"), errors="coerce")
    strategy["applied_date"] = pd.to_datetime(strategy.get("activity_start"), errors="coerce")
    strategy["sample_size"] = 1

    strategy = strategy.rename(columns={"name": "restaurant_name"})
    strategy_cols = [
        c for c in [
            "cluster_id", "cluster_label", "strategy_name", "restaurant_name", "restaurant_id", "channel",
            "applied_date", "bookings_before", "bookings_after", "bookings_uplift_pct",
            "revenue_before", "revenue_after", "revenue_uplift_pct",
            "incremental_revenue_thb", "roi", "sample_size", "activity_id"
        ] if c in strategy.columns
    ]
    strategy = strategy[strategy_cols].copy()
else:
    strategy = pd.DataFrame(columns=[
        "cluster_id", "cluster_label", "strategy_name", "restaurant_name", "restaurant_id", "channel",
        "applied_date", "bookings_before", "bookings_after", "bookings_uplift_pct",
        "revenue_before", "revenue_after", "revenue_uplift_pct",
        "incremental_revenue_thb", "roi", "sample_size", "activity_id"
    ])

strategy_path = export_dir / "cluster_strategy_outcomes.parquet"
strategy.to_parquet(strategy_path, index=False)
print(f"Saved: {strategy_path} ({len(strategy):,} rows)")
